# Fixing "Retrieval algorithm for rainfall mapping from microwave links in a cellular communication network" pycomlink implementation

url: https://amt.copernicus.org/articles/9/2425/2016/amt-9-2425-2016.pdf

## 3 Methodology

Table 2. Most important variables used in the (sub)functions of the rainfall retrieval algorithm.

|  Name in (sub)function | Symbol in text | Unit | Description  |
| --- | --- | --- | --- |
|  a | a | $mm h^{-1} dB^{-b} km^{b}$ | Coefficient of R-k power law  |
|  $A_{max}$ | $A_{max}$ | dB | Maximum rain-induced attenuation  |
|  $A_{min}$ | $A_{min}$ | dB | Minimum rain-induced attenuation  |
|  b | b | - | Exponent of R-k power law  |
|  DateTime | NA | UTC | Date and time  |
|  Dry | NA | - | Should interval be considered dry for reference level determination? (0 = wet; 1 = dry)  |
|  F | F | $dB km^{-1} h$ | Computed for filter to remove outliers  |
|  Frequency | f | GHz | Microwave frequency  |
|  ID | ID | - | Unique link identifier  |
|  PathLength | L | km | Path length  |
|  $R_{mean}$ | ⟨R⟩ | $mm h^{-1}$ | Path-averaged rainfall intensity  |
|  $P_{min}$ | $P_{min}$ | dB | Minimum received power  |
|  $P_{min}Cor$ | $P^{C}_{min}$ | dB | Corrected minimum received power  |
|  $P_{max}$ | $P_{max}$ | dB | Maximum received power  |
|  $P_{max}Cor$ | $P^{C}_{max}$ | dB | Corrected maximum received power  |
|  $P_{ref}$ | $P_{ref}$ | dB | Reference level  |
|  $X_{Start}$ | NA | ° (km) | Longitude (or easting) of start of microwave link  |
|  $X_{End}$ | NA | ° (km) | Longitude (or easting) of end of microwave link  |
|  $Y_{Start}$ | NA | ° (km) | Latitude (or northing) of start of microwave link  |
|  $Y_{End}$ | NA | ° (km) | Latitude (or northing) of end of microwave link  |

### 3.1 Classification of wet and dry periods: the nearby link approach

In order to define wet and dry periods, it is assumed that rain is correlated in space and hence that several links in a given area should experience a joint decrease in received signal level in the case of rain. A time interval is labelled as wet if at least half of the links in the vicinity (default radius is 15 km, but this can be modified to better match other time intervals) of the selected link experience such a decrease. A detailed description of this classification algorithm can be found in Appendix C.

Note that this processing step is optional. The user can also decide not to apply a wet-dry classification, which may be the only option in areas with low spatial link densities.

#### Appendix C: Wet-dry classification with nearby link approach

A step-by-step description of the classification algorithm (function “WetDryNearbyLinkApMinMaxRSL”) is given below and mainly obtained from Overeem et al. (2011). The classification is run for each link for the period for which data are provided. Note that running this wet-dry classification is optional.

1. The link coordinates are converted to an azimuthal equidistant cartesian coordinate system (easting and northing of start of link, easting and northing of end of link; km).
2. Select a link.
3. All links for which both end points are within a chosen radius (default 15 km) from either end of the already selected link are selected as well.
4. Continue if at least three surrounding links have been selected for the considered time interval for which the link in step 2 has data, otherwise the link is not used for that time interval. If the link is part of a full-duplex link, the other link is counted as surrounding link.
5. Calculate the attenuation $ΔP = P_{min} − max(P_{min})$ ($P_{min} = RSL_{min}$) and specific attenuation $ΔPL = \frac{P_{min} − max(P_{min})}{L}$ for each link and each time interval. $max(P_{min})$ is the maximum value of $P_{min}$ over the previous number of hours including the present time interval (default 24 h). Note that $max(P_{min})$ is only computed if at least a minimum number of hours of data are available (default 6 h); otherwise it is not computed and no rainfall intensities will be retrieved.
6. The median values of ΔP and ΔPL are computed over all selected links for each time interval.
7. If median(ΔPL) < −0.7 dB km⁻¹ and median(ΔP) < −1.4 dB the time interval is classified as wet for the link selected in step 1.
8. If $max(P_{min}) − P_{min} > 2 dB$ for a given time interval that is classified as wet, the previous two time intervals and the next time interval are classified as wet for the link selected in step 1.

9. All time intervals that have not been classified as wet are classified as dry for the link selected in step 1.
10. Repeat these steps for all other links.

Note that a radius of 10 km is used in Overeem et al. (2011), whereas a default radius of 15 km is used here and in Overeem et al. (2013), which allows estimating rainfall in areas with lower spatial link densities. The 15 km radius is representative of the decorrelation distance of convective rainfall in the Netherlands in case of a time interval of 15 min. For the often occurring stratiform rainfall this distance will be (much) longer. Step 4 has been slightly altered with respect to Overeem et al. (2011).

The threshold values in steps 7 and 8 have been obtained from Overeem et al. (2011), who optimise them by visual comparison with the gauge-adjusted radar data set of path-averaged rainfall intensities employing data from 2009 (NEC links with 0.1 dB power resolution). Step 8 was used because rainfall is generally correlated in time, and sometimes very local, so that it does not occur at surrounding links.

If the algorithm would be rerun for another period, for instance one time interval later, this can result in different rainfall estimates compared to the preceding run. This is due to step 8 of the wet-dry classification methodology, which may also classify the previous 30 min as rainy. In order to obtain the same rainfall estimates for different runs, the algorithm would need to be slightly modified or one should wait 30 min before applying the algorithm. Another option is to not apply step 8, which can be supplied as function argument. The rainfall retrieval algorithm is suitable for real-time application, for which the reclassification of previous time intervals is of no consequence, because rainfall intensity of present time interval is of interest.

The wet/dry classification algorith can be found at pycomlink: https://github.com/pycomlink/pycomlink/blob/master/pycomlink/processing/wet_dry/nearby_wetdry.py

Does the wet/dry algorithm exclude the link itself?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

- ⏳ `web_search(type="search", queries="['site:github.com/pycomlink/pycomlink ne…", query="site:github.com/pycomlink/pycomlink near…")` ⏳


```json {.tool}
{
  "id": "ws_046bc0d016c8f623006aad3f8e6d2087d19874c1052911eadf",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:github.com/pycomlink/pycomlink nearby_wetdry.py WetDryNearbyLinkApMinMaxRSL"
    ],
    "query": "site:github.com/pycomlink/pycomlink nearby_wetdry.py WetDryNearbyLinkApMinMaxRSL"
  },
  "result": "Server tool call executed.",
  "server": true
}
```


- ⏳ `web_search(type="open_page", url="https://raw.githubusercontent.com/pycoml…")` ⏳


```json {.tool}
{
  "id": "ws_046bc0d016c8f623006aad3f923bc487d1b3cb5ee37052f9e6",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://raw.githubusercontent.com/pycomlink/pycomlink/master/pycomlink/processing/wet_dry/nearby_wetdry.py"
  },
  "result": "Server tool call executed.",
  "server": true
}
```

No. In the `pycomlink` implementation, the selected link is **included** in its own nearby-link set.

The reason is explicit in the distance calculation: distances between a link’s own endpoints are set to zero. Consequently, its diagonal entry satisfies the radius test, and `ds_nearby_cmls` includes the selected link itself. ([raw.githubusercontent.com](https://raw.githubusercontent.com/pycomlink/pycomlink/master/pycomlink/processing/wet_dry/nearby_wetdry.py))
[*](https://raw.githubusercontent.com/pycomlink/pycomlink/master/pycomlink/processing/wet_dry/nearby_wetdry.py "")
This matters for the threshold:

```python
if sum(ds_dist.within_r.sel(cml_id1=cmlid).values) > min_links:
```

With the default `min_links=3`, the condition requires more than three links **including the selected link**—normally at least four links total, equivalent to at least three other nearby links. The median calculations also operate over that set, so the target link contributes to `medianP` and `medianPL`. ([raw.githubusercontent.com](https://raw.githubusercontent.com/pycomlink/pycomlink/master/pycomlink/processing/wet_dry/nearby_wetdry.py))

So the[*](https://raw.githubusercontent.com/pycomlink/pycomlink/master/pycomlink/processing/wet_dry/nearby_wetdry.py "") practical answer is:

- **Does it exclude itself?** No.
- **Does it require three surrounding links by default?** Yes, effectively, because the target link counts as one of the four-or-more links.
- **Does the target link influence the median?** Yes.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 49225, "completion_tokens": 565, "total_tokens": 49790, "cached_tokens": 8774, "reasoning_tokens": 302, "cost": 0.013415519999999998}
```


### 3.3 Filter to remove outliers

Malfunctioning links can cause outliers in rainfall retrievals. These outliers can be removed by using a filter that is based on the assumption that rainfall is correlated in space. The filter discards a time interval of a link for which the cumulative difference between its specific attenuation and that of the surrounding links (i.e. within a default radius of 15 km) over the previous 24 h (default value; including the present time interval) becomes lower than the outlier filter threshold (dB km$^{-1}$ h$^{-1}$). This criterion is applied to specific attenuation derived from uncorrected minimum received power (Overeem et al., 2013). Imagine that the default value of 32.5 dB km$^{-1}$ h$^{-1}$ is uniformly distributed over all time intervals in a 24 h period. This implies a maximum specific attenuation of approximately 1.35 dB km$^{-1}$ (32.5 dB km$^{-1}$ h$^{-1}$ divided by 24 h) per time interval. This corresponds to a daily rain accumulation of approximately 120 mm for a 38.9 GHz link and 750 mm for the least sensitive, 13 GHz, link. Hence, a time interval of a chosen link will only be discarded if the rainfall amounts during the previous 24 h period are substantial. It is therefore highly unlikely that this filter would discard real rain.

The value of the cumulative difference, F, is computed as follows:

$$F = \sum_{t=-24 \text{ h} + \Delta t}^{0} \left( \Delta P_{\text{L,t}}^{\text{S}} - \text{median}(\Delta P_{\text{L,t}}) \right) \Delta t, \tag{2}$$

where t is the time interval, t = 0 being the present time interval for which F needs to be computed, and Δt is the time interval in hours (0.25 h in the working example). A link is not used to estimate rainfall if F < F$_{t}$. Note that $ΔP_{L}^{S}$, median(ΔP$_{L}$), and F are computed in the nearby link approach and are based on the minimum received powers, the superscript S referring to the selected link for which rainfall is to be computed. Running the outlier filter is optional.

### 3.x Q/A

How do they define a link?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

A **link** is one microwave communication path between two endpoints—typically two cellular network antennas—over which the received signal level is measured.

In this algorithm, each link is identified by an `ID` and described by:

- the coordinates of its start and end points;
- its path length \(L\) in kilometres;
- its operating frequency \(f\);
- minimum and maximum received powers, \(P_{\min}\) and \(P_{\max}\), for each time interval.

The rainfall estimate is therefore a **path-averaged** value along the line between the two antennas, not a point measurement. In a full-duplex connection, the two transmission directions may appear as separate links, although the opposite direction is counted as a surrounding link during wet–dry classification.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 44613, "completion_tokens": 221, "total_tokens": 44834, "reasoning_tokens": 57, "cost": 0.0137817}
```


So I'm getting a bit confused by the naming. I'm used to the [OpenSense V2 standard](https://open-research-europe.ec.europa.eu/articles/3-169) where: 
- (cml_id) Link identifier: A string uniquely identifying a link in a network. It associates time series in the raw data with link features in the metadata.
- (sublink_id) Sublink identifier: a string that identifies the sublink channel. The term sublink is used to distinguish among all the possible signals travelling between the same pair of antennas.

Can you help me understand the naming they used in the paper?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The paper uses **“link”** and **“link path”** differently:

| Paper term | Meaning | Closest OpenSense concept |
|---|---|---|
| **Link path** | The physical/geometric path between two antennas, independent of direction | Closest to a physical **CML/link** |
| **Link** | A radio connection in one direction, from one tower to another | Closest to a directional **sublink/channel** |
| **ID** | The unique identifier for that directional link time series | Closest to a directional sublink identifier |

The crucial sentence is in the introduction:

> Many links are full duplex … in which case we have two links but only one link path.

So their model is:

```text
antenna A  ─────────  antenna B
             one link path

A → B        one link, one ID
B → A        one link, another ID
```

The two directional links share the same physical path, but each has its own time series and identifier. Appendix A makes this explicit: a full-duplex link should have **two IDs, one for each link direction**.

That means the paper’s `ID` is **not** an identifier for the physical path. It identifies the individual directional measurement series. In OpenSense terminology, it is therefore closer to a `sublink_id` than to a `cml_id`, assuming `cml_id` groups both directions between the same antenna pair.

The paper does not use the term **sublink**. Its terminology is effectively:

```text
link path ≈ physical antenna-pair connection
link       ≈ directional channel/measurement
ID         ≈ identifier of that directional measurement
```

This distinction also explains the interpolation description in #_a1fc8da4: full-duplex links are averaged when they have the same midpoint. The algorithm first treats them as separate directional links, then combines their rainfall estimates when converting link-path observations into point observations.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 62973, "completion_tokens": 933, "total_tokens": 63906, "cached_tokens": 61007, "reasoning_tokens": 516, "cost": 0.0040994099999999995}
```


## Niger

In [ ]:
# !pip install shapely pandas geopandas xarray folium pycomlink raincell

In [ ]:
from pathlib import Path
import sys

from shapely import box, Point, LineString
import pandas as pd
import geopandas as gpd
import xarray as xr
import folium

from pycomlink.processing.wet_dry import nearby_wetdry
from pycomlink.processing import k_R_relation as kR
from raincell.maps import setup_default_map, explore_sublinks
from raincell.maps._vector import explore_gauges

*You need to adapt the code bellow to point to the right DATA_DIR*. Probably the simplest it to do:
```python
DATA_DIR = Path("...")
```


In [ ]:
sys.path.insert(0, "./SetGMDI_validation")
from config import DATA_DIR

In [ ]:
COUNTRY = "niger"
RADIUS_KM = 2

### Helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from raincell.plot._romulo import reduce_cml_id_dim

def romulo_plot_cml(
        cml: xr.DataArray, # CML data to plot
        title: str = None, # Title for the plot
        fig: plt.Figure = None, 
        ax: plt.Axes = None, 
        figsize: tuple = (16, 9), # Figure size, only used if fig or ax are None
        vmax: float | None = None # Maximum value for colormap
        ) -> tuple[plt.Figure, plt.Axes]:
    """Plot an overview of the CML data sorted by length. Negative values, which are not phisically possible are shown in red."""
    cml = cml.copy()
    if len(cml.dims) > 2:
        cml = reduce_cml_id_dim(cml)
    cml_sorted = cml.sortby("length")

    cmap = plt.cm.Blues.copy()
    cmap.set_under('lightgrey')

    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    cml_sorted.plot(ax=ax, x="time", vmin=0, vmax=vmax,cmap=cmap, extend='min')

    ax.set_yticks([])
    ax.set_ylabel("Sublink data sorted by length")
    ax.set_title(title or cml.name)

    return fig, ax

### Color definition

In [ ]:
cmaps = {
    "niger": {
        "cmap": "gist_rainbow", "vmin": 17000, "vmax": 23000, "imerg_early": "#ff7777ff", "imerg_late": "#ff0000ff", "gauge": "#000000"
        },
}

In [ ]:
from functools import partial

def get_color(v, cmap:str, vmin: float, vmax: float, gauge: str, imerg_early: str, imerg_late: str):
    if isinstance(v, str):
        if "gauge" in v.lower():
            return gauge
        elif "imerg early" in v.lower():
            return imerg_early
        elif "imerg late" in v.lower():
            return imerg_late
    else:
        norm = Normalize(vmin=vmin, vmax=vmax)
        cmap = plt.get_cmap(cmap)
        return cmap(norm(v))


In [ ]:
get_color = partial(
    get_color, 
    cmap=cmaps[COUNTRY]["cmap"], 
    vmin=cmaps[COUNTRY]["vmin"], 
    vmax=cmaps[COUNTRY]["vmax"],
    imerg_early=cmaps[COUNTRY]["imerg_early"],
    imerg_late=cmaps[COUNTRY]["imerg_late"],
    gauge=cmaps[COUNTRY]["gauge"]
) 

### Open and select data

In [ ]:
def get_fps(country: list) -> dict:
    def get_first_nc(d: Path) -> Path: return list(d.glob("*.nc"))[0]
    
    return {
            "cml": get_first_nc(DATA_DIR/f"{country}/L0/"),
            "gauge": get_first_nc(DATA_DIR/f"{country}/reference/"),
            "imerg_early_v07": get_first_nc(DATA_DIR/f"{country}/aux/imerg/early_v07/"),
            "imerg_late_v07": get_first_nc(DATA_DIR/f"{country}/aux/imerg/late_v07/")
            }

In [ ]:
fps = get_fps(COUNTRY)

In [ ]:
urban_bounds = {
    "niger": {"niamey": dict(lon=(1.95, 2.25), lat=(13.65, 13.45))},
}

In [ ]:
bounds = urban_bounds[COUNTRY]
bounds = bounds[list(bounds)[0]]
bounds

{'lon': (1.95, 2.25), 'lat': (13.65, 13.45)}

In [ ]:
def get_bbox_gdf(lats, lons):
    return gpd.GeoDataFrame(geometry=[box(lons[0], lats[0], lons[1], lats[1])], crs="EPSG:4326")

In [ ]:
bbox = get_bbox_gdf(bounds["lat"], bounds["lon"])

In [ ]:
cml = xr.load_dataset(fps["cml"])
cml = cml.reindex(time=pd.date_range(cml.time.min().item(), cml.time.max().item(), freq='15min'))
gauge = xr.load_dataset(fps["gauge"])
imerg_early = xr.open_dataset(fps["imerg_early_v07"]).sel(lat=slice(*bounds["lat"]), lon=slice(*bounds["lon"])).load()
imerg_late = xr.load_dataset(fps["imerg_late_v07"]).sel(lat=slice(*bounds["lat"]), lon=slice(*bounds["lon"])).load()


/app/data/.local/lib/python3.12/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'gribberish' loading failed:
No module named 'obstore'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [ ]:
round_prec = 1 # set to 0.5 if you want 0.5 increments 1, 1.5, ...
cml["link_frequency"] = (cml.frequency.mean("sublink_id") / 1e3 / round_prec).round() * round_prec
cml = cml.set_coords("link_frequency")

In [ ]:
def links_to_gdf(cml: xr.Dataset) -> gpd.GeoDataFrame:
    links = cml.cml_id.to_dataframe().reset_index(drop=True)
    g = [LineString([[r.site_0_lon, r.site_0_lat], [r.site_1_lon, r.site_1_lat]]) for _, r in links.iterrows()]
    return gpd.GeoDataFrame(links.drop(columns=["site_0_lat", "site_1_lat", "site_0_lon", "site_1_lon"]), geometry=g, crs="EPSG:4326")

In [ ]:
def get_cml_in_bbox(cml, bbox):
    links = links_to_gdf(cml)
    return cml.sel(cml_id=gpd.sjoin(links, bbox).cml_id.unique())

In [ ]:
cml = get_cml_in_bbox(cml, bbox)

In [ ]:
def gauge_to_gdf(gauge: xr.Dataset) -> gpd.GeoDataFrame:
    df = gauge.id.to_dataframe().reset_index(drop=True)
    return gpd.GeoDataFrame(df.drop(columns=["lat", "lon"]), geometry=[Point(r["lon"], r["lat"]) for _, r in df.iterrows()], crs="EPSG:4326")

In [ ]:
def get_gauge_in_bbox(gauge, bbox):
    gdf = gauge_to_gdf(gauge)
    return gauge.sel(id=gpd.sjoin(gdf, bbox).id.unique())

In [ ]:
gauge = get_gauge_in_bbox(gauge, bbox)

In [ ]:
def get_cml_near_gauges(cml, gauge, radius_km):
    gauges = gauge_to_gdf(gauge)
    links = links_to_gdf(cml)

    # Choose a local projected CRS automatically.
    metric_crs = links.estimate_utm_crs()

    gauges_m = gauges.to_crs(metric_crs)
    links_m = links.to_crs(metric_crs)

    # One circle per gauge, with radius in metres.
    circles = gauges_m.copy()
    circles["geometry"] = circles.geometry.buffer(radius_km*1e3)

    # Keep links intersecting at least one circle.
    matches = gpd.sjoin(
        links_m,
        circles[["geometry"]],
        how="inner",
        predicate="intersects",
    )

    return cml.sel(cml_id=matches.cml_id.unique())

In [ ]:
cml = get_cml_near_gauges(cml, gauge, radius_km=RADIUS_KM)

In [ ]:
def add_gauge_influence_radius(
    gauge,
    radius_km: int = 5,
    m: folium.Map = None,
    color: str = "black",
    opacity: float = 0.1
):
    m = m or folium.Map()
    circle_layer = folium.FeatureGroup(name=f"{radius_km} km gauge radius")

    for _, g in gauge.id.to_dataframe().iterrows():
        folium.Circle(
            location=[g["lat"], g["lon"]],
            radius=radius_km*1e3,
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=opacity,
            tooltip=f"{g['id']} — {radius_km} km radius",
        ).add_to(circle_layer)

    circle_layer.add_to(m)
    return m

In [ ]:
m = explore_sublinks(cml, add_transmission_sense=False, default=None, **cmaps[COUNTRY])
# m = bbox.explore(m=m, color="black", style_kwds=dict(fillOpacity=0), hover=False)
m = add_gauge_influence_radius(gauge, radius_km=RADIUS_KM, m=m)
# m = gauge_pixel_matches.explore(m=m, color="green", style_kwds=dict(fillOpacity=0.3))
# m = get_imerg_centers(imerg_early).explore(m=m)
# m = links_to_gdf(cml).explore(m=m, color="red")
m = explore_gauges(gauge, default=None, m=m)
# m = gauge_to_gdf(gauge).explore(m=m, color="black")
setup_default_map(m)

In [ ]:
romulo_plot_cml(-cml.rsl_min.where(cml.rsl_min.notnull(), 1))

## Fixed version of the function including old code and comments

The main issue of the current code is that it uses the links for neighbour computation while it shoud use the sublinks. This is proably a naming issue as the paper uses "link" for what the OpenSense standard calls "sublink". So the code is adapted to use sublinks for neighbour computation.

### 1. Variable definition and distance computation
The link coordinates are converted to an azimuthal equidistant cartesian coordinate system (easting and northing of start of link, easting and northing of end of link; km).

In [ ]:
pmin=cml.rsl_min.copy() # Time series of pmin must include cml_id and time as dimensions.

src code:

```python
ds_dist=endpoint_dist.copy() # Distance matrix between all CML endpoints calculated with `calc_distance_between_cml_endpoints`
```

In [ ]:
endpoint_dist = nearby_wetdry.calc_distance_between_cml_endpoints(
    cml_ids=cml.cml_id.values,
    site_a_latitude=cml.site_0_lat.values,
    site_a_longitude=cml.site_0_lon.values,
    site_b_latitude=cml.site_1_lat.values,
    site_b_longitude=cml.site_1_lon.values
)

0it [00:00, ?it/s]

24it [00:00, 1579.38it/s]

In [ ]:
dist=endpoint_dist.copy()

We replace ds_dist by dist in order to be consistent with pmin. It will be clear that is a dataset from type annotation.

In [ ]:
radius=15 # Radius for which surrounding CMLs are considered for the wet-dry detection (both end points have to be within the chosen radius from either end of the selected link).
thresh_median_P=-1.4 # Threshold for median_P. Is dependent on the spatial correlation of       rainfall.
thresh_median_PL=-0.7 # Threshold for median_PL. Is dependent on the spatial correlation of rainfall.

src code:

```python
min_links=3 # minimum number of CMLs within radius needed to perform wet-dry classification.
```

In [ ]:
min_nearby_sublinks=3 # minimum number of nearby sublinks (excluding the selected sublink) within radius needed to perform wet-dry classification.


I think its better to replace link by sublink as its closer to the OpenSense standard naming.

In [ ]:
interval=15 #  Interval of pmin in minutes.
timeperiod=24 # Number of previous hours over which max(pmin) should be computed.
min_hours=6 # Minimum number of hours needed to compute max(pmin).

### X. Stack the sublinks

We will stack the sublinks in order to have a single `id` dim. This will simplify a lot the following code regarding neighbouring sublink checks.

In [ ]:
def stack_sublinks(da: xr.DataArray | xr.Dataset, sep: str = " | ") -> xr.DataArray | xr.Dataset:
    stacked = da.stack(id=("cml_id", "sublink_id"))
    labels = [f"{c}{sep}{s}" for c, s in zip(stacked.cml_id.values, stacked.sublink_id.values)]
    stacked = stacked.reset_index("id")
    stacked = stacked.assign_coords(id=("id", labels))
    has_data = stacked.notnull().any(dim="time")
    if isinstance(has_data, xr.Dataset):
        has_data = has_data.to_array().any("variable")
    return stacked.sel(id=has_data)

In [ ]:
pmin_stck = stack_sublinks(pmin)
pmin_stck.dims

### 5. 

Calculate the attenuation $ΔP = P_{min} − max(P_{min})$ ($P_{min} = RSL_{min}$) and specific attenuation $ΔPL = \frac{P_{min} − max(P_{min})}{L}$ for each link and each time interval. $max(P_{min})$ is the maximum value of $P_{min}$ over the previous number of hours including the present time interval (default 24 h). Note that $max(P_{min})$ is only computed if at least a minimum number of hours of data are available (default 6 h); otherwise it is not computed and no rainfall intensities will be retrieved.

#### 5.1

$max(P_{min})$

src code:

```python
# rolling window * 60min/interval(in minutes)
period = int(timeperiod * 60 / interval)
```

In [ ]:
# rolling window in minutes for max(Pmin) computation
period = int(timeperiod * 60 / interval)
period

src code:

```python
# min hours for calculation * 60min/interval(in minutes)
hours_needed = int(min_hours * 60 / interval)
```

In [ ]:
# minimum widow size rolling window in minutes for max(Pmin) computation
min_periods = int(min_hours * 60 / interval)
min_periods

The code silently truncates non-integral windows:

```python
period = int(timeperiod * 60 / interval)
min_periods = int(min_hours * 60 / interval)
```

For example, an interval of seven minutes cannot represent either six or 24 hours exactly. Personally I think this is not a real issue as usual sampling times (1min, 5min, 15min, ...) will produce integral windows. But it is worth noting and we might need to discuss it.

src code:

```python
max_pmin = pmin.rolling(time=period, min_periods=hours_needed).max(skipna=False)
```

The source code as it is assumes that the data is has regular time intervals and there are not missing time intervals. However this is not always the case and I think the user should be warned.

In [ ]:
def check_regular_intervals(
    cml, # cml data in OpenSense standard
    interval: int # expected time interval
    ):
    time = cml.time.to_index()
    tstep = time.diff()[1:].unique()
    msg = f"""Expected a regular {interval}-minute time coordinate. xarray rolling mode will give misleading results as window is
    set through int and not through string/timedelta. Please regularize the time index before passing the data
    to the function:
    - e.g. `da.reindex(time=pd.date_range(da.time.min(), da.time.max(), freq='{interval}min'))`: inserts NaN at missing steps.  Use when timestamps already align to a fixed grid/sampling rate but some are absent.
    - e.g. `da.resample(time='{interval}min').mean(skipna=True)`: aggregates values into regular bins. Use when timestamps don't fall on a fixed grid/sampling rate.
    """
    if tstep.size != 1 or tstep[0] != pd.Timedelta(f"{interval}min"):
        raise ValueError(msg)

In [ ]:
check_regular_intervals(pmin.isel(time=[0, 20, 30]), interval)

In [ ]:
max_pmin = pmin_stck.rolling(time=period, min_periods=min_periods).max(skipna=True)

Rename for clarity. I did some experiments (I've removed them for clarity but they core idea can be seen in the AI response below). It turns out that for xarray’s rolling maximum, **skipna=False** does not make the whole window invalid when it contains missing values. **min_periods** controls the required number of valid observations. Thus **skipna=False** is misleading so its set to **True** so the behaviour is clear. IMO its not very clear if this is the expected behaviour from the algorithm described on the paper...

I've done few tests using skipna=True and skipna=False at its returning the same result. Can you search on the internet for clarification?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I’ll check the official xarray documentation and implementation notes for rolling reductions and `skipna`, then reconcile them with your observed output.
- ⏳ `web_search(type="search", queries="['site:docs.xarray.dev rolling max skipn…", query="site:docs.xarray.dev rolling max skipna …")` ⏳


```json {.tool}
{
  "id": "ws_0fcd31ccdda9d4ba006abd157bc07087d1a19f2eddacb36c35",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:docs.xarray.dev rolling max skipna xarray DataArrayRolling max",
      "site:github.com/pydata/xarray DataArrayRolling max skipna rolling_reduce"
    ],
    "query": "site:docs.xarray.dev rolling max skipna xarray DataArrayRolling max"
  },
  "result": "Server tool call executed.",
  "server": true
}
```


- ⏳ `web_search(type="search", queries="['site:github.com/pydata/xarray/blob/mai…", query="site:github.com/pydata/xarray/blob/main/…")` ⏳


```json {.tool}
{
  "id": "ws_0fcd31ccdda9d4ba006abd1583842c87d1b73060bce12173d5",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:github.com/pydata/xarray/blob/main/xarray/core/rolling.py \"def max\" skipna",
      "site:github.com/pydata/xarray/blob/main/xarray/core/rolling.py \"skipna\" \"DataArrayRolling\"",
      "site:docs.xarray.dev/en/stable/generated/xarray.DataArrayRolling.max.html"
    ],
    "query": "site:github.com/pydata/xarray/blob/main/xarray/core/rolling.py \"def max\" skipna"
  },
  "result": "Server tool call executed.",
  "server": true
}
```


- ⏳ `web_search(type="open_page", url="https://raw.githubusercontent.com/pydata…")` ⏳


```json {.tool}
{
  "id": "ws_0fcd31ccdda9d4ba006abd15889be487d1ade14ba130d9f5b5",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://raw.githubusercontent.com/pydata/xarray/main/xarray/computation/rolling.py"
  },
  "result": "Server tool call executed.",
  "server": true
}
```


- ⏳ `web_search(type="open_page", url="https://raw.githubusercontent.com/pydata…")` ⏳


```json {.tool}
{
  "id": "ws_0fcd31ccdda9d4ba006abd158a445887d1b4349f80057d3a51",
  "name": "web_search",
  "args": {
    "type": "open_page",
    "url": "https://raw.githubusercontent.com/pydata/xarray/main/xarray/computation/rolling.py"
  },
  "result": "Server tool call executed.",
  "server": true
}
```

You’re right—my previous explanation was incorrect for **`xarray.rolling(...).max()`**.

In your example, both settings produce the same result because rolling reductions use **`min_periods` to decide whether a window is valid**. With `min_periods=1`, every window containing at least one valid value is eligible, and the rolling maximum ignores the missing values in practice. Xarray’s optimized rolling implementation uses a `min_count`/valid-observation count internally rather than treating `skipna=False` as “reject any window containing a NaN.” ([docs.xarray.dev](https://docs.xarray.dev/en/v2024.05.0/generated/xarray.DataArray.rolling.html?utm_source=openai))
[*](https://docs.xarray.dev/en/v2024.05.0/generated/xarray.DataArray.rolling.html?utm_source=openai "xarray.DataArray.rolling")
To see the actual distinction between the reduction behaviours, construct the windows explicitly:

```python
windows = x.rolling(time=3, min_periods=1).construct("window")

print(windows.values)
print(windows.max("window", skipna=True).values)
print(windows.max("window", skipna=False).values)
```

The windows are:

```text
[
    [nan, nan, nan],
    [nan, nan, 2],
    [nan, 2, nan],
    [2, nan, 1],
]
```

The direct reductions give:

```text
skipna=True   → [nan, 2, 2, 2]
skipna=False  → [nan, nan, nan, nan]
```

So the earlier intuition about `skipna` applies to the **direct reduction**:

```python
windows.max("window", skipna=False)
```

but not straightforwardly to the optimized rolling call:

```python
x.rolling(time=3, min_periods=1).max(skipna=False)
```

For the pycomlink call:

```python
pmin.rolling(
    time=period,
    min_periods=min_periods,
).max(skipna=False)
```

with:

```python
period = 96       # trailing 24 hours
min_periods = 24  # at least 6 hours of valid observations
```

the practical interpretation is:

> `max_pmin` is the maximum available `pmin` within the trailing 96 samples, provided that at least 24 valid observations are present.

Therefore, `min_periods=24` is the setting that implements the “at least six hours of data” rule. `skipna=False` does **not** make the entire 24-hour window invalid when it contains one missing sample. Xarray documents `min_periods` as the minimum number of observations required in a rolling window, and its current rolling source applies that count separately from the reduction itself. ([docs.xarray.dev](https://docs.xarray.dev/en/v2024.05.0/generated/xarray.DataArray.rolling.html?utm_source=openai))
[*](https://docs.xarray.dev/en/v2024.05.0/generated/xarray.DataArray.rolling.html?utm_source=openai "xarray.DataArray.rolling")
A compact test of the pycomlink logic is:

```python
x.rolling(time=3, min_periods=2).max(skipna=False).values
```

Here the last window `[2, nan, 1]` has two valid observations, so it produces `2`; a window with fewer than two valid observations produces `nan`.

```json {.usage}
{"model": "gpt-5.6-luna", "prompt_tokens": 84806, "completion_tokens": 2259, "total_tokens": 87065, "cached_tokens": 51841, "reasoning_tokens": 1560, "cost": 0.015510929999999997}
```


In [ ]:
romulo_plot_cml(-max_pmin)

#### 5.2

$ΔP = P_{min} − max(P_{min})$

src code

```python
deltaP = pmin - max_pmin
```

In [ ]:
deltaP = pmin_stck - max_pmin

Update pmin to use the stacked version

In [ ]:
romulo_plot_cml(-deltaP.where(deltaP.notnull(), 1))

#### 5.3

$ΔP_L = \frac{P_{min} − max(P_{min})}{L}$ 

src code:

```python
deltaPL = deltaP / (pmin.length / 1e3)
```

In [ ]:
deltaPL = deltaP / (pmin_stck.length / 1e3)

Update pmin to use the stacked version. Also, assume that we are using the OpenSense standard and that length must be converted from meters to km.

In [ ]:
romulo_plot_cml(-deltaPL.where(deltaPL.notnull(), 1))

### 3. 

All links for which both end points are within a chosen radius (default 15 km) from either end of the already selected link are selected as well.

We create a mask to know which sublinks are nearby of each sublink

src code

```python
# get number of CMLs within radius for each CML
ds_dist["within_r"] = (
    (ds_dist.a_to_all_a < radius)
    & (ds_dist.a_to_all_b < radius)
    & (ds_dist.b_to_all_a < radius)
    & (ds_dist.b_to_all_b < radius)
)
```

In [ ]:
# get number of CMLs within radius for each CML
within_r = (
    (dist.a_to_all_a < radius)
    & (dist.a_to_all_b < radius)
    & (dist.b_to_all_a < radius)
    & (dist.b_to_all_b < radius)
)

Replace ds_dist by dist to match the new name. A new variable is created instead of assigning it to dist. Otherwise global dist will be modified which is an unexpected side effect IMO.

In [ ]:
from matplotlib.colors import ListedColormap

within_r_int = within_r.astype(int)

im = within_r_int.plot.imshow(
    figsize=(10, 8),
    cmap=ListedColormap(["white", "steelblue"]),
    vmin=0,
    vmax=1,
    interpolation="none",
    cbar_kwargs={"ticks": [0, 1], "label": "Within radius"},
)

ax = im.axes
ax.set(
    xlabel="CML ID 2",
    ylabel="CML ID 1",
    title="CML links within radius",
)
ax.tick_params(axis="x", labelrotation=90)

src code:

```python
ds_cml = pmin.to_dataset(name="pmin")
ds_cml["max_pmin"] = max_pmin
ds_cml["deltaP"] = deltaP
ds_cml["deltaPL"] = deltaPL
```

I don't think ds_cml is useful. Its only used for selection but I think this selection can be made more clear even thought if has to be done twice. In addition its never returned so I think its better to remove it.

src code:

```python
wet = xr.full_like(ds_cml.pmin, np.nan)
F = xr.full_like(ds_cml.pmin, np.nan)
```

In [ ]:
wet = xr.full_like(pmin, np.nan, dtype=float)
F = xr.full_like(pmin, np.nan, dtype=float)

Its simpler to declare them directly from pmin. In addition If **pmin** is an integer array, it cannot represent **NaN** correctly. Received signal data will normally be floating point, but it could happen for 1dB quantization to be integer, so making this explicit is safer.

src code:

```python
medianP_out = xr.full_like(ds_cml.pmin, np.nan)
medianPL_out = xr.full_like(ds_cml.pmin, np.nan)
```

This variables are declared but never used so I think its better to remove them for clarity

### 2. Select a link. 

1st loop

src code:

```python
for cmlid in tqdm(ds_cml.cml_id):
```

In [ ]:
# for idx, cmlid, sublinkid in tqdm(zip(pmin_stck.id, pmin_stck.cml_id, pmin_stck.sublink_id)):
idx, cmlid, sublinkid = next(iter(zip(pmin_stck.id, pmin_stck.cml_id, pmin_stck.sublink_id)))

With the stacked version we should access to the three ids as this will simplify the code downstream

In [ ]:
idx, cmlid, sublinkid

### 4. 

Continue if at least three surrounding links have been selected for the considered time interval for which the link in step 2 has data, otherwise the link is not used for that time interval. If the link is part of a full-duplex link, the other link is counted as surrounding link.##

src code:

```python
if sum(ds_dist.within_r.sel(cml_id1=cmlid).values) > min_links:
```

The original if will check the sourrounding links and not sublinks as in the original paper. Thus a new version is proposed. We will add few helping variables

In [ ]:
nearby_cml_ids = within_r.cml_id2[within_r.sel(cml_id1=cmlid)]

nearby_sublinks = pmin_stck.id[pmin_stck.cml_id.isin(nearby_cml_ids)]

Then checking sourrounding sublinks is straightforward. We will use it for the if statement.

In [ ]:
nearby_sublinks.size > min_nearby_sublinks

True

### 3.

Back to nearby sublink selection

src code:

```python
# only make wet dry detection if min_links is reached within radius
# select all CMLs within r
ds_nearby_cmls = ds_cml.isel(cml_id=ds_dist.within_r.sel(cml_id1=cmlid).values)
```

In [ ]:
nearby_dP = deltaP.sel(id=nearby_sublinks)
nearby_dPL = deltaPL.sel(id=nearby_sublinks)

So selecting the required deltaP and deltaPL is quiet simple (and as said before there is no need for ds_cml).

### 6. The median values of ΔP and ΔPL are computed over all selected links for each time interval.

src code:

```python
# calculate median delta P and delta PL for each timestep
# .where checks if the minimal require number of CMLs has data
medianP = ds_nearby_cmls.deltaP.where(ds_nearby_cmls.deltaP.count("cml_id") > min_links)
medianP = medianP.median(dim="cml_id", skipna=True).values
```

In [ ]:
medianP = nearby_dP.where(nearby_dP.count("id") > min_nearby_sublinks)
medianP = medianP.median(dim="id", skipna=True)

So counting is done based on the number of sublinks instead of the number of links. values access is removed as its not necessary.

In [ ]:
tsel = dict(time=slice(None, "2017-07-06"))
deltaP.sel(id=idx).sel(**tsel).plot()
medianP.sel(**tsel).plot(alpha=0.6)

src code:

```python
medianP_out.loc[dict(cml_id=cmlid)] = medianP
```

As explained before this vaiable is never used so it should be removed.

src code:

```python
medianPL = ds_nearby_cmls.where(ds_nearby_cmls.deltaP.count("cml_id") > min_links)
medianPL = medianPL.deltaPL.median(dim="cml_id", skipna=True).values
```

In [ ]:
medianPL = nearby_dPL.where(nearby_dPL.count("id") > min_nearby_sublinks)
medianPL = medianPL.median(dim="id", skipna=True)

In [ ]:
tsel = dict(time=slice(None, "2017-07-06"))
deltaPL.sel(id=idx).sel(**tsel).plot()
medianPL.sel(**tsel).plot(alpha=0.6)

src code:

```python
medianPL_out.loc[dict(cml_id=cmlid)] = medianPL
```

### 7. 

If median(ΔPL) < −0.7 dB km⁻¹ and median(ΔP) < −1.4 dB the time interval is classified as wet for the link selected in step 1.

src code:

```python
# actual wet dry classification
wet.loc[dict(cml_id=cmlid)] = xr.where(
    cond=(np.isnan(medianP)) | (np.isnan(medianPL)),
    x=np.nan,
    y=((medianP < thresh_median_P) & (medianPL < thresh_median_PL)),
)
```

In [ ]:
wet_sublink = ((medianP < thresh_median_P) & (medianPL < thresh_median_PL))
wet_sublink = wet_sublink.where(medianP.notnull() & medianPL.notnull())

Personally I find this easier to read. In addition we remove the need of assigning the data every time to `wet` which is our final output. 

There is a contradition between this implementation and the step 9 in the paper but it done in purpose. Step 9 states that:
> All time intervals that have not been classified as wet are classified as dry.
So:
```
1      wet
0      dry, with sufficient valid data
NaN    unclassified or unavailable
```
However in the current implementation periods with insufficient data remain NaN, rather than being classified as dry.
I think that the implementation is the correct behaviour but is should be discussed.

In [ ]:
tsel = dict(time=slice(None, "2017-07-06"))
fig, ax1 = plt.subplots(figsize=(8, 4))
alpha = 0.6
# Left axis: ΔP [dB]
medianP.sel(**tsel).plot(
    ax=ax1, color="tab:blue", alpha=alpha,
    label=r"median $\Delta P$",
)
ax1.axhline(
    thresh_median_P, color="tab:blue",
    linestyle="--", alpha=alpha,
    label=fr"$\Delta P$ threshold ({thresh_median_P})",
)
ax1.set_ylabel(r"$\Delta P$ [dB]", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

# Right axis: ΔP_L [dB/km]
ax2 = ax1.twinx()

medianPL.sel(**tsel).plot(
    ax=ax2, color="tab:orange", alpha=alpha,
    label=r"median $\Delta P_L$",
)
ax2.axhline(
    thresh_median_PL, color="tab:orange",
    linestyle="--", alpha=alpha,
    label=fr"$\Delta P_L$ threshold ({thresh_median_PL})",
)
ax2.set_ylabel(r"$\Delta P_L$ [dB/km]", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

ax1.grid(alpha=0.3)

wet_sublink_ss = wet_sublink.sel(**tsel)

for t, value in zip(wet_sublink_ss.time.values, wet_sublink_ss.values):
    if np.isfinite(value) and value == 1:
        start = pd.Timestamp(t)
        end = start + pd.Timedelta(minutes=interval)
        ax1.axvspan(
            start,
            end,
            color="lightgrey",
            alpha=0.35,
            zorder=-1,
        )

# Combine legends from both axes
lines = ax1.lines + ax2.lines
ax1.legend(lines, [line.get_label() for line in lines], loc="lower center")

### (src) Calculate the F which is used for the outlier filter

src code:

```python
# calculate the F values which can be used as outlier filter
F_val = ds_nearby_cmls.deltaPL.sel(cml_id=cmlid) - medianPL
F.loc[dict(cml_id=cmlid)] = F_val.rolling(time=timeperiod * 4).sum(skipna=True) * (interval / 60)
```

Its better to move this filter to the end so wet computation and F computation are not tangled

### 8. 

If $max(P_{min}) − P_{min} > 2 dB$ for a given time interval that is classified as wet, the previous two time intervals and the next time interval are classified as wet for the link selected in step 1.

src code:

```python
wet_tmp = wet.copy()
```

```python
# If wet is true (condition 1) and deltaP < -2db  (condition 2)
# then set the two time steps before and the one after the
# considered time step to wet.
```

```python
for shift in [1, -1, -2]:
    wet.loc[dict(cml_id=cmlid.values)] = xr.where(
        (
            (wet_tmp.loc[dict(cml_id=cmlid)] == 1)  # condition 1
            & (ds_nearby_cmls.sel(cml_id=cmlid).deltaP < -2)  # condition 2
        ).shift(
            time=shift  # shift here
        ),
        x=1,  # if condition 1 and 2 are true, set value to 1 (wet)
        y=wet.loc[dict(cml_id=cmlid.values)],  # else, keep value
    )
# this adds nans at all time steps where the shift overwrote
# previous nans
wet.loc[dict(cml_id=cmlid.values)] = xr.where(
    np.isnan(wet_tmp.loc[dict(cml_id=cmlid.values)]),
    np.nan,
    wet.loc[dict(cml_id=cmlid.values)],
)
```

In [ ]:

# The mask will be used to filter data "created" during the strong att expansion bellow
valid_wet = wet_sublink.notnull() # we need to create the mask before the variable is modif.
valid_deltaP = deltaP.sel(id=idx).notnull()

is_wet = wet_sublink.notnull() & wet_sublink.astype(bool)
deltaP_2dB = deltaP.sel(id=idx) < -2
for shift in [1, -1, -2]:
    wet_sublink = xr.where(
        (is_wet & deltaP_2dB).shift(time=shift),
        x=1,  # if condition 1 and 2 are true, set value to 1 (wet)
        y=wet_sublink,  # else, keep value
    )

# this adds nans at all time steps where the shift overwrote
# previous nans and where pmin is nan valid
wet.loc[dict(cml_id=cmlid, sublink_id=sublinkid)] = wet_sublink.where(valid_wet & valid_deltaP)

While this code is longer its IMHO easier to understand. It will fix the cml_id based assignation and it will assign each sublink separately (as expected). In addition, mask generation is done once before the loop and not on every iteration. Same is true for wet (output) assignation which is done at the end and only once per sublink.

### 9. and 10. 
9. All time intervals that have not been classified as wet are classified as dry for the link selected in step 1. *Its done during the previous steps when creating the wet variable*
10. Repeat these steps for all other links. *It corresponds to the loop*

### (new) Calculate the F which is used for the outlier filter

The filter discards a time interval of a link for which the cumulative difference between its specific attenuation and that of the surrounding links (i.e. within a default radius of 15 km) over the previous 24 h (default value; including the present time interval) becomes lower than the outlier filter threshold (dB km$^{-1}$ h$^{-1}$). This criterion is applied to specific attenuation derived from uncorrected minimum received power (Overeem et al., 2013).

The value of the cumulative difference, F, is computed as follows:

$$F = \sum_{t=-24 \text{ h} + \Delta t}^{0} \left( \Delta P_{\text{L,t}}^{\text{S}} - \text{median}(\Delta P_{\text{L,t}}) \right) \Delta t, \tag{2}$$

where t is the time interval, t = 0 being the present time interval for which F needs to be computed, and Δt is the time interval in hours (0.25 h in the working example). A link is not used to estimate rainfall if F < F$_{t}$. Note that $ΔP_{L}^{S}$, median(ΔP$_{L}$), and F are computed in the nearby link approach and are based on the minimum received powers, the superscript S referring to the selected link for which rainfall is to be computed. Running the outlier filter is optional.

We replace the original `timeperiod * 4` by `period` which shoud be more consistent with the previous code.

src code:

```python
# calculate the F values which can be used as outlier filter
F_val = ds_nearby_cmls.deltaPL.sel(cml_id=cmlid) - medianPL
F.loc[dict(cml_id=cmlid)] = F_val.rolling(time=timeperiod * 4).sum(skipna=True) * (interval / 60)
```

In [ ]:
# calculate the F values which can be used as outlier filter
dt = interval / 60
f = (deltaPL.sel(id=idx) - medianPL).rolling(time=period).sum(skipna=True) * dt
F.loc[dict(cml_id=cmlid, sublink_id=sublinkid)] = f

deltaPL is used directly as ds_nearby_cmls does not exist anymore. Each sublink is assigned on each loop. And dt is renamed for clarity.

This might be the expected behaviour as we are computing a sum but in this case there is no a minimum number of values required to compute the sum (as it was the case for *max*(${p_{min}}$)). So for example if there is one single 15min valid interval the sum will still be computed.

### X. Rename variables and assign minimum attributes

In [ ]:
wet.name = "wet"
wet.attrs = {"long_name": "wet_dry_classification_based_on_overeem_et_al_2016"}

In [ ]:
F.name = "F"
F.attrs = {"long_name": "cummulative_diff_filter_based_on_overeem_et_al_2016"}

To avoid confussion variables are renamed and long name assigned.

### Wrap up

Other improvements to the function and the doc:

- Rename `nearby_wetdry` to `calc_nearby_wetdry`. The reason is that `nearby_wetdry` is both the name of the module and the package, which creates conflicts. In addition, the general Python recommendation is to use a verb in function names so that it is clear what the function does. I used `calc` for consistency with `calc_distance_between_cml_endpoints`.
- Add the type annotations directly to the function signature. Personally, I think this is clearer and has some advantages if type checking is introduced in the future.
- Remove the defaults and types from the documentation. They are redundant and more error-prone.
- Change the type from `xr.Dataset` to `xr.DataArray`. The type in the original function is incorrect.
- `pmin` requires `cml_id`, `sublink_id`, and `time`, not only `cml_id` and `time`.
- `dist` is accessed through `cml_id1` and `cml_id2`; saying only that it “must include `cml_id`” is vague so I removed it.
- The documentation calls the output an “F-score”, but \(F\) is a cumulative difference/filter quantity, not generally an F-score. I've renamed as F cumulative difference but there might be a better name

In [ ]:
def calc_nearby_wetdry(
    pmin: xr.DataArray,
    dist: xr.Dataset,
    radius: float = 15,
    thresh_median_P: float = -1.4,
    thresh_median_PL: float = -0.7,
    min_nearby_sublinks: int = 3,
    interval: int = 15,
    timeperiod: int = 24,
    min_hours: int = 6,
) -> tuple[xr.DataArray, xr.DataArray]:
    """
    Classification of rainy and dry periods from diagnostic (min-max) CML signal
    levels following the nearby link approach from Overeem et al. (2016).
    OpenSense standard is assumed and variable names are adopted from the paper.
    In case of mismatch (e.g. link/sublink definition), OpenSense is used.

    Parameters
    ----------
    pmin :
         Time series of pmin in OpenSense standard.
    dist :
         Distance matrix between all CML endpoints calculated with
         `calc_distance_between_cml_endpoints()`.
    radius : 
        Radius for which surrounding CMLs are considered for the wet-dry
        detection (both end points have to be within the chosen radius from
        either end of the selected link).
    thresh_median_P : 
        Threshold for median_P. Is dependent on the spatial correlation of
        rainfall.
    thresh_median_PL : 
        Threshold for median_PL. Is dependent on the spatial correlation of
        rainfall.
    min_nearby_sublinks : 
        minimum number of sublinks (excluding the selected sublink) within 
        radius needed to perform wet-dry classification.
    interval: 
         Interval of pmin in minutes.
    timeperiod : 
        Number of previous hours over which max(pmin) should be computed.
    min_hours :
        Minimum number of hours needed to compute max(pmin).

    Returns
    -------
        Time series of wet-dry classification and F cumulative difference used 
        for quality control.

    References
    ----------
    .. [1] Overeem, A., Leijnse, H., and Uijlenhoet, R.: Retrieval algorithm
    for rainfall mapping from microwave links in a cellular communication
    network, Atmos. Meas. Tech., 9, 2425–2444,
    https://doi.org/10.5194/amt-9-2425-2016, 2016.
    """
    def check_regular_intervals(
        cml: xr.DataArray | xr.Dataset, # cml data in OpenSense standard
        interval: int # expected time interval in minutes
        ):
        time = cml.time.to_index()
        tstep = time.diff()[1:].unique()
        msg = f"""Expected a regular {interval}-minute time coordinate. xarray rolling mode will give misleading results as window is
        set through int and not through string/timedelta. Please regularize the time index before passing the data
        to the function:
        - e.g. `da.reindex(time=pd.date_range(da.time.min(), da.time.max(), freq='{interval}min'))`: inserts NaN at missing steps.  Use when timestamps already align to a fixed grid/sampling rate but some are absent.
        - e.g. `da.resample(time='{interval}min').mean(skipna=True)`: aggregates values into regular bins. Use when timestamps don't fall on a fixed grid/sampling rate.
        """
        if tstep.size != 1 or tstep[0] != pd.Timedelta(f"{interval}min"):
            raise ValueError(msg)

    check_regular_intervals(pmin, interval)

    # Stack cml_id and sublink_id into a single id dim. All the coordinates are kept.
    pmin_stck = pmin.stack(id=("cml_id", "sublink_id"))
    labels = [f"{c} | {s}" for c, s in zip(pmin_stck.cml_id.values, pmin_stck.sublink_id.values)]
    pmin_stck = pmin_stck.reset_index("id")
    pmin_stck = pmin_stck.assign_coords(id=("id", labels))
    has_data = pmin_stck.notnull().any(dim="time")
    pmin_stck = pmin_stck.sel(id=has_data)

    # rolling window * 60min/interval(in minutes)
    period = int(timeperiod * 60 / interval)

    # min hours for calculation * 60min/interval(in minutes)
    min_periods = int(min_hours * 60 / interval)

    max_pmin = pmin_stck.rolling(time=period, min_periods=min_periods).max(skipna=False)

    deltaP = pmin_stck - max_pmin

    deltaPL = deltaP / (pmin_stck.length / 1e3) # convert len to km

    # get number of CMLs within radius for each CML
    within_r = (
        (dist.a_to_all_a < radius)
        & (dist.a_to_all_b < radius)
        & (dist.b_to_all_a < radius)
        & (dist.b_to_all_b < radius)
    )

    # Initialize the output ds that will be filled on the loop
    wet = xr.full_like(pmin, np.nan)
    F = xr.full_like(pmin, np.nan)

    for idx, cmlid, sublinkid in tqdm(zip(pmin_stck.id, pmin_stck.cml_id, pmin_stck.sublink_id)):

        nearby_cml_ids = within_r.cml_id2[within_r.sel(cml_id1=cmlid)]

        nearby_sublinks = pmin_stck.id[pmin_stck.cml_id.isin(nearby_cml_ids)]

        if nearby_sublinks.size > min_nearby_sublinks: #make wet dry detection if min_nearby_sublinks is reached within radius
            # select all sublinks within r
            nearby_dP = deltaP.sel(id=nearby_sublinks)
            nearby_dPL = deltaPL.sel(id=nearby_sublinks)

            # calculate median delta P and delta PL for each timestep
            # .where checks if the minimal require number of sublinks has data at each timestep
            medianP = nearby_dP.where(nearby_dP.count("id") > min_nearby_sublinks)
            medianP = medianP.median(dim="id", skipna=True)

            medianPL = nearby_dPL.where(nearby_dPL.count("id") > min_nearby_sublinks)
            medianPL = medianPL.median(dim="id", skipna=True)

            wet_sublink = ((medianP < thresh_median_P) & (medianPL < thresh_median_PL))
            wet_sublink = wet_sublink.where(medianP.notnull() & medianPL.notnull())

            # The mask will be used to filter data "created" during the strong att expansion bellow
            valid_wet = wet_sublink.notnull() # we need to create the mask before the variable is modif.
            valid_deltaP = deltaP.sel(id=idx).notnull()

            is_wet = wet_sublink.notnull() & wet_sublink.astype(bool)
            deltaP_2dB = deltaP.sel(id=idx) < -2
            for shift in [1, -1, -2]:
                wet_sublink = xr.where(
                    (is_wet & deltaP_2dB).shift(time=shift),
                    x=1,  # if condition 1 and 2 are true, set value to 1 (wet)
                    y=wet_sublink,  # else, keep value
                )

            # this adds nans at all time steps where the shift overwrote
            # previous nans and where pmin is nan valid
            wet.loc[dict(cml_id=cmlid, sublink_id=sublinkid)] = wet_sublink.where(valid_wet & valid_deltaP)

            # calculate the F values which can be used as outlier filter
            dt = interval / 60
            Fval = (deltaPL.sel(id=idx) - medianPL).rolling(time=period).sum(skipna=True) * dt
            F.loc[dict(cml_id=cmlid, sublink_id=sublinkid)] = Fval

    wet.name = "wet"
    wet.attrs = {"long_name": "wet_dry_classification_based_on_overeem_et_al_2016"}

    F.name = "F"
    F.attrs = {"long_name": "cumulative_diff_filter_based_on_overeem_et_al_2016"}
    return wet, F

## Apply wet/dry classification

In [ ]:
endpoint_dist = nearby_wetdry.calc_distance_between_cml_endpoints(
    cml_ids=cml.cml_id.values,
    site_a_latitude=cml.site_0_lat.values,
    site_a_longitude=cml.site_0_lon.values,
    site_b_latitude=cml.site_1_lat.values,
    site_b_longitude=cml.site_1_lon.values
)

0it [00:00, ?it/s]

24it [00:00, 1744.27it/s]

In [ ]:
wetdry, F_flag = calc_nearby_wetdry(cml.rsl_min, dist=endpoint_dist)

0it [00:00, ?it/s]

5it [00:00, 40.63it/s]

10it [00:00, 41.37it/s]

15it [00:00, 41.56it/s]

20it [00:00, 41.72it/s]

25it [00:00, 41.57it/s]

30it [00:00, 41.36it/s]

35it [00:00, 41.44it/s]

38it [00:00, 41.44it/s]

In [ ]:
romulo_plot_cml(wetdry.where(wetdry.notnull(), -1))
romulo_plot_cml(wetdry.where(wetdry.notnull(), -1).sel(time=slice("2017-08-21", "2017-08-24")))

In [ ]:
wetdry_stck = stack_sublinks(wetdry)
romulo_plot_cml(wetdry_stck.where(wetdry_stck.notnull(), -1).sel(time=slice("2017-08-21", "2017-08-24")))